# AI Video Generator API Backend (for Desktop App)

**Instructions:**
1. Go to `Runtime` -> `Change runtime type` and ensure `T4 GPU` is selected.
2. Go to `Runtime` -> `Run all`.
3. Scroll to the bottom and wait for a green link that looks like `https://xxxxxx.gradio.live`.
4. Copy that link and paste it into your Desktop App!

In [ ]:
!pip install -q diffusers transformers accelerate gradio imageio[ffmpeg] decord peft

In [ ]:
import torch
import gradio as gr
from diffusers import LTXPipeline, StableVideoDiffusionPipeline
from diffusers.utils import export_to_video, load_image
from PIL import Image
import os

# Variables to hold our models in memory
current_model_name = None
pipeline = None

def load_model(model_choice):
    global current_model_name, pipeline
    
    if current_model_name == model_choice:
        return pipeline
        
    # Clear memory
    if pipeline is not None:
        del pipeline
        torch.cuda.empty_cache()
        
    print(f"Loading {model_choice}...")
    
    if "LTX-Video" in model_choice:
        pipeline = LTXPipeline.from_pretrained(
            "Lightricks/LTX-2.3",
            torch_dtype=torch.bfloat16
        )
        # Load LoRA if selected
        if "LoRA" in model_choice:
            print("Loading LoRA weights...")
            pipeline.load_lora_weights("Lora-Daddy/Ltx2.3-real-nudity-early-alpha-30k-steps")
        
        pipeline.enable_model_cpu_offload()
    elif "Stable Video Diffusion" in model_choice:
        pipeline = StableVideoDiffusionPipeline.from_pretrained(
            "stabilityai/stable-video-diffusion-img2vid-xt", 
            torch_dtype=torch.float16, 
            variant="fp16"
        )
        pipeline.enable_model_cpu_offload()
        
    current_model_name = model_choice
    return pipeline

def generate_video(image_path, prompt, model_choice):
    pipe = load_model(model_choice)
    output_path = "output_video.mp4"
    
    if "LTX-Video" in model_choice:
        if not prompt:
            prompt = "A beautiful cinematic shot"
            
        video_frames = pipe(
            prompt=prompt,
            negative_prompt="worst quality, blurry",
            width=704,
            height=480,
            num_frames=121,
            num_inference_steps=50,
        ).frames[0]
        
        export_to_video(video_frames, output_path, fps=24)
        
    elif "Stable Video Diffusion" in model_choice:
        if not image_path:
            raise gr.Error("Image is required for SVD")
            
        # Load and resize image for SVD
        img = load_image(image_path)
        img = img.resize((1024, 576))
        
        frames = pipe(img, decode_chunk_size=8, generator=torch.manual_seed(42)).frames[0]
        export_to_video(frames, output_path, fps=7)
        
    return output_path

# Create the Gradio interface (API only needed for desktop app, but UI helps debugging)
iface = gr.Interface(
    fn=generate_video,
    inputs=[
        gr.Image(type="filepath", label="Input Image"),
        gr.Textbox(label="Prompt"),
        gr.Radio([
            "Stable Video Diffusion (Image-to-Video)", 
            "LTX-Video 2.3 (Text-to-Video)",
            "LTX-Video 2.3 + Nudity LoRA (Text-to-Video)"
        ], label="Model", value="Stable Video Diffusion (Image-to-Video)")
    ],
    outputs=gr.Video(label="Generated Video")
)

print("Starting API Server...")
iface.launch(share=True)